# MEG Connectivity Analysis - Complete Example

Step-by-step all-to-all source-space connectivity analysis (DICS beamformer,
`conpy`) for the Spatial-Temporal Working Memory MEG project.

Every heavy step is delegated to a function in
`STWM_functions_for_connectivity.py`; each of those functions receives the
single `config` dictionary, exactly as in `STWM_functions_core.py`.

**Workflow**
- *Group prerequisites* (run once for the whole sample): build the fsaverage
  template (`src_average`), morph it to every subject, build forward models,
  and identify the common connectivity pairs (`pairs_identification`).
- *Individual analysis* (per subject): morph, forward model, DICS connectivity
  estimation, and visualization of the contrast.
- *Group statistics* (run once): cluster-permutation test and rendering.

**Requirements**
- Completed sensor-space analysis (CSD files `<subject>_<condition>_csd.h5`)
- FreeSurfer reconstruction + coregistration (`<subject>-trans.fif`)
- `conpy` package: `pip install conpy`

## Setup and Imports

In [ ]:
import os
import yaml
import numpy as np
import mne
import matplotlib.pyplot as plt

# Import the individual-analysis driver and the connectivity functions
from connectivity_individual_analysis import run_connectivity_analysis, load_config
from STWM_functions_for_connectivity import (
    src_average,
    new_morphing,
    new_morphed_forward_model,
    pairs_identification,
    connectivity_estimation,
    connectivity_vizualization,
    connectivity_statistics,
    connectivity_statistics_visualization
)

print("✓ All modules imported successfully")
print(f"MNE version: {mne.__version__}")

# For interactive 3D brain visualization use 'qt' (requires PyQt5);
# for inline plotting use 'inline'.
%matplotlib inline
# %matplotlib qt

## Load Configuration

In [ ]:
# Load configuration
config_path = 'config.yaml'
config = load_config(config_path)

# Select the subject to analyse
config['subject']['subject_name'] = 'S1'

subject_name = config['subject']['subject_name']
condition_1 = config['conditions']['condition_1']['name']
condition_2 = config['conditions']['condition_2']['name']
freq_min = config['connectivity']['freq_min']
freq_max = config['connectivity']['freq_max']
spacing = config['connectivity']['spacing']

print(f"Subject     : {subject_name}")
print(f"Conditions  : {condition_1} vs {condition_2}")
print(f"Band        : {freq_min}-{freq_max} Hz")
print(f"Spacing     : {spacing}")

### Prerequisites Check

In [ ]:
def check_prerequisites(config):
    """Check that the required files for connectivity analysis exist."""
    folder = config['paths']['data_folder']
    subjects_dir = config['paths']['subjects_dir']
    subject_name = config['subject']['subject_name']
    condition_1 = config['conditions']['condition_1']['name']
    condition_2 = config['conditions']['condition_2']['name']

    print("Checking prerequisites...\n")

    # FreeSurfer subject directory
    fs_subj_dir = os.path.join(subjects_dir, subject_name)
    print(("✓" if os.path.exists(fs_subj_dir) else "✗"),
          f"FreeSurfer directory: {fs_subj_dir}")

    # Coregistration file
    trans_file = os.path.join(folder, f"{subject_name}-trans.fif")
    print(("✓" if os.path.exists(trans_file) else "✗"),
          f"Coregistration file: {trans_file}")

    # CSD files from the sensor-space analysis
    for cond in (condition_1, condition_2):
        csd = os.path.join(folder, f"{subject_name}_{cond}_csd.h5")
        print(("✓" if os.path.exists(csd) else "✗"),
              f"CSD file: {csd}")

    # Group-level pairs (needed for estimation)
    pairs = os.path.join(folder, "Average-pairs.npy")
    print(("✓" if os.path.exists(pairs) else "✗"),
          f"Connectivity pairs: {pairs} (group-level, see Steps 0 & 3)")

    try:
        import conpy  # noqa: F401
        print("✓ conpy package available")
    except ImportError:
        print("✗ conpy package NOT available  ->  pip install conpy")

check_prerequisites(config)

---
## Group prerequisites (run ONCE for the whole sample)

The next three steps build the common reference shared by all subjects. They
only need to be executed once; afterwards, individual subjects reuse the saved
files. Skip ahead to *Step 1* if these have already been produced.

### Step 0: Build the fsaverage Template Source Space

`src_average(config)` creates the template (`Sub_for_con_Avg-<spacing>-src.fif`)
and checks that vertices fall within sensor range. Uses
`connectivity.average_ref_file` and `connectivity.average_trans_file`.

In [ ]:
print(f"Building fsaverage template source space ({spacing})...")
try:
    fsaverage = src_average(config)
    print(f"\n✓ Template source space created")
    print(f"  Left hemisphere : {fsaverage[0]['nuse']} sources")
    print(f"  Right hemisphere: {fsaverage[1]['nuse']} sources")
except Exception as e:
    print(f"\n✗ Error: {e}")
    fsaverage = None

### Step 3 (group): Identify Connectivity Pairs

After every subject has a morphed forward model (Steps 1-2 below, run for all
subjects), `pairs_identification(config)` selects the vertices shared across
subjects and builds the all-to-all pairs saved as `Average-pairs.npy`. Set
`config['connectivity']['num_subjects']` accordingly.

In [ ]:
print("Identifying connectivity pairs across subjects...")
print(f"  num_subjects = {config['connectivity']['num_subjects']}")
try:
    pairs = pairs_identification(config)
    print(f"\n✓ Connectivity pairs created: {len(pairs[0])} pairs")
except Exception as e:
    print(f"\n✗ Error: {e}")
    pairs = None

---
## Individual-level analysis (per subject)

These steps are exactly what `connectivity_individual_analysis.py` runs. You can
either execute them cell by cell below, or run the whole pipeline in one call:

```python
run_connectivity_analysis(config)
```

### Step 1: Morph the Source Space to the Subject

`new_morphing(config)` morphs the fsaverage template onto the subject anatomy
and saves `<subject>_for_con-morph-src.fif`.

In [ ]:
print(f"Morphing template source space to {subject_name}...")
try:
    subject_src = new_morphing(config)
    print(f"\n✓ Morphed source space created")
    print(f"  Left hemisphere : {subject_src[0]['nuse']} sources")
    print(f"  Right hemisphere: {subject_src[1]['nuse']} sources")
except Exception as e:
    print(f"\n✗ Error: {e}")
    subject_src = None

### Step 2: Create the Morphed Forward Model

`new_morphed_forward_model(config)` restricts the morphed source space to
vertices within sensor range and computes the forward solution
(`<subject>-for_con-morphed-fwd.fif`).

In [ ]:
print(f"Creating forward model for {subject_name}...")
try:
    fwd = new_morphed_forward_model(config)
    print(f"\n✓ Forward model created")
    print(f"  Sources used: {fwd['nsource']}")
    print(f"  Channels    : {fwd['nchan']}")
except Exception as e:
    print(f"\n✗ Error: {e}")
    fwd = None

### Step 4: Estimate DICS Connectivity

`connectivity_estimation(config)` loads the CSD matrices, averages over the
band, and computes all-to-all DICS connectivity for both conditions, saving one
connectivity object per condition.

In [ ]:
print(f"Estimating connectivity ({freq_min}-{freq_max} Hz)...")
try:
    connectivity_1, connectivity_2 = connectivity_estimation(config)
    print(f"\n✓ Connectivity estimated for '{condition_1}' and '{condition_2}'")
    print(f"  {condition_1}: {connectivity_1[0]}")
    print(f"  {condition_2}: {connectivity_2[0]}")
except Exception as e:
    print(f"\n✗ Error: {e}")
    connectivity_1 = connectivity_2 = None

### Step 5: Visualize the Connectivity Contrast

`connectivity_vizualization(config)` renders the contrast (condition 1 -
condition 2) as a parcellated circle plot and on the cortical surface.

In [ ]:
print(f"Visualizing connectivity contrast for {subject_name}...")
try:
    p, brain = connectivity_vizualization(config)
    print(f"\n✓ Visualization created")
except Exception as e:
    print(f"\n⚠ Visualization failed: {e}")
    p = brain = None

---
## Group-level statistics (run ONCE across subjects)

After connectivity has been estimated for every subject (Step 4 looped over all
subjects), the following steps test the condition contrast at the group level.

### Step 6: Cluster-Permutation Statistics

`connectivity_statistics(config)` averages the per-subject connectivity onto
fsaverage and runs `conpy.cluster_permutation_test`, saving the surviving
connections and an HDF5 file with the cluster details.

In [ ]:
print("Running connectivity cluster-permutation statistics...")
print(f"  Subjects     : {config['connectivity']['num_subjects']}")
print(f"  Permutations : {config['connectivity']['n_permutations']}")
print(f"  Threshold    : {config['connectivity']['cluster_threshold']}")
try:
    (connection_indices, bundles, bundle_ts,
     bundle_ps, H0, contrast) = connectivity_statistics(config)
    print(f"\n✓ Statistics complete")
    print(f"  Surviving connections: {len(connection_indices)}")
    print(f"  Bundles              : {len(bundle_ps)}")
except Exception as e:
    print(f"\n✗ Error: {e}")

### Step 7: Visualize the Statistical Result

`connectivity_statistics_visualization(config)` renders the significant
contrast as a parcellated circle plot (optionally restricted via `regexp`) and
on the cortical surface.

In [ ]:
print("Rendering group statistical connectivity...")
try:
    brain_stat, con_parc = connectivity_statistics_visualization(config)
    print(f"\n✓ Statistical visualization created")
except Exception as e:
    print(f"\n⚠ Visualization failed: {e}")
    brain_stat = con_parc = None

## Notes

- **Subject naming**: connectivity steps assume subjects are named `S1`, `S2`,
  ... (the estimation/statistics functions index files as `S<i>-...`).
- **Run order**: Step 0 -> (Steps 1-2 for every subject) -> Step 3 ->
  (Step 4 for every subject) -> Steps 6-7. Steps 1-5 for a single subject are
  exactly `run_connectivity_analysis(config)`.
- **Parameters** live in the `connectivity` section of `config.yaml`; see
  `example_usage_connectivity.py` for a fully documented configuration.
- **conpy** must be installed: `pip install conpy`.